# Exploring EntailmentBank

EntailmentBank (Dalvi et al., EMNLP 2021) ships **gold multi-step reasoning**:
each item is a hypothesis plus an entailment tree that derives it from numbered
facts. So there is nothing to generate -- the proof *is* the reasoning chain.

This notebook looks at the raw data and at how this repo turns each tree into a
linear reasoning string (`reasoning_trajectories.datasets.entailmentbank`).

Data: unzipped `entailment_trees_emnlp2021_data_v3` at
`../data/raw/entailment_trees_emnlp2021_data_v3/` (already present).
Needs `pandas` + `matplotlib`.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from reasoning_trajectories.datasets.entailmentbank import (
    EntailmentBank, build_id2text, parse_proof_steps, linearize_proof,
)

EB_ROOT = Path.cwd().parent / "data" / "raw" / "entailment_trees_emnlp2021_data_v3"
DATA = EB_ROOT / "dataset"
pd.set_option("display.max_colwidth", 140)

def read_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]

## 1. Tasks and splits

Same 1,840 trees three ways: **task_1** = only the gold leaf sentences,
**task_2** = gold leaves + 25 distractors, **task_3** = retrieve from a 12k
corpus (no proof stored in the file -- the loader borrows the task_1 tree).

In [ ]:
splits = ["train", "dev", "test"]
raw = {t: {s: read_jsonl(DATA / f"task_{t}" / f"{s}.jsonl") for s in splits} for t in (1, 2, 3)}
pd.DataFrame({f"task_{t}": {s: len(raw[t][s]) for s in splits} for t in (1, 2, 3)})

In [ ]:
rec = raw[1]["dev"][0]
print("top-level keys:", list(rec))
print("meta keys:     ", list(rec["meta"]))
rec

## 2. Tree shape

`depth_of_proof` = longest root-to-leaf path, `length_of_proof` = number of
entailment steps (internal nodes).

In [ ]:
def summarize(records, task):
    rows = []
    for r in records:
        m = r.get("meta") or {}
        rows.append({
            "id": r["id"],
            "depth": r.get("depth_of_proof"),
            "length": r.get("length_of_proof"),
            "n_leaves": len(m.get("triples") or {}),
            "n_intermediate": len(m.get("intermediate_conclusions") or {}),
            "n_distractors": len(m.get("distractors") or []),
            "q_words": len((r.get("question") or m.get("question_text") or "").split()),
            "hyp_words": len((r.get("hypothesis") or "").split()),
            "task": task,
        })
    return pd.DataFrame(rows)

df1 = summarize(raw[1]["train"] + raw[1]["dev"] + raw[1]["test"], 1)
df2 = summarize(raw[2]["train"] + raw[2]["dev"] + raw[2]["test"], 2)
df1[["depth", "length", "n_leaves", "n_intermediate", "hyp_words"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.3))
df1["depth"].value_counts().sort_index().plot.bar(ax=axes[0], title="proof depth (task_1)")
df1["length"].value_counts().sort_index().plot.bar(ax=axes[1], title="proof length / #steps")
df1["n_leaves"].value_counts().sort_index().plot.bar(ax=axes[2], title="# leaf facts")
df2["n_distractors"].value_counts().sort_index().plot.bar(ax=axes[3], title="# distractors (task_2)")
for a in axes:
    a.tick_params(axis="x", labelrotation=0)
fig.tight_layout()

## 3. Entailment-tree processing

`proof` is a linear string like
`"sent2 & sent3 -> int1: <text>; int1 & sent1 -> hypothesis;"`.
The repo resolves every `sentN` / `intN` id (via `meta.triples` /
`meta.intermediate_conclusions`, falling back to parsing `context`) and
linearizes the tree.

In [ ]:
# pick a deeper tree to show multi-step output
deep = max(raw[1]["dev"], key=lambda r: r.get("length_of_proof") or 0)
print("id:", deep["id"], "| depth:", deep["depth_of_proof"], "| steps:", deep["length_of_proof"])
print("\nraw proof string:\n ", deep["proof"])
print("\nparsed steps:")
for s in parse_proof_steps(deep["proof"]):
    print("  ", s)

In [ ]:
for fmt in ("natural", "structured", "raw"):
    text, steps = linearize_proof(deep, fmt=fmt)
    print(f"===== {fmt} =====")
    print(text)
    print()

## 4. The repo loader -> `Example`

`EntailmentBank` yields `Example` objects: a `prompt` (facts + hypothesis,
ending at `"Reasoning:"`) and `meta["reasoning"]` = the linearized gold chain,
which `reasoning.source: dataset_field` picks up. `validation` maps to `dev`.

In [ ]:
ds1 = EntailmentBank(task=1, split="validation", local_dir=EB_ROOT, shuffle_seed=None)
ex = next(e for e in ds1 if len(e.meta["entailment_steps"]) >= 3)
print(ex.prompt)
print("-" * 80)
print("GOLD REASONING (meta['reasoning']):\n")
print(ex.meta["reasoning"])

In [ ]:
# task_2 = same tree, but the facts list now includes 25 sentences (gold + distractors)
ds2 = EntailmentBank(task=2, split="validation", local_dir=EB_ROOT, shuffle_seed=None)
ex2 = next(e for e in ds2 if e.id == ex.id)
print(ex2.prompt.split("Statement:")[0])
print("n_distractors:", ex2.meta["n_distractors"])

In [ ]:
# task_3 = no proof in the file -> loader borrows the task_1 tree
ds3 = EntailmentBank(task=3, split="validation", local_dir=EB_ROOT, shuffle_seed=None)
ex3 = next(e for e in ds3 if e.id == ex.id)
print("proof_borrowed_from_task1:", ex3.meta["proof_borrowed_from_task1"])
print(ex3.meta["reasoning"])

## 5. How long are the gold chains?

The linearized chain is what gets tokenised and traced, so its size sets the
number of points on the token trajectory.

In [ ]:
rows = [{"id": e.id, "steps": len(e.meta["entailment_steps"]),
         "reasoning_words": len(e.meta["reasoning"].split())}
        for e in EntailmentBank(task=1, split="validation", local_dir=EB_ROOT, shuffle_seed=None)]
rdf = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
rdf["reasoning_words"].hist(bins=30, ax=axes[0]); axes[0].set_title("linearized chain length (words)")
rdf.plot.scatter(x="steps", y="reasoning_words", ax=axes[1], alpha=0.4); axes[1].set_title("words vs #steps")
fig.tight_layout()
rdf.describe()

In [ ]:
# optional: exact token counts with the model tokenizer
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B")
    n = [len(tok(e.meta["reasoning"]).input_ids)
         for e in EntailmentBank(task=1, split="validation", local_dir=EB_ROOT, shuffle_seed=None)]
    print(pd.Series(n, name="reasoning_tokens").describe())
except Exception as e:
    print("skipped (tokenizer unavailable):", e)

### Next steps

- Extract hidden states over these gold chains:
  `python ../scripts/run_extraction.py --config ../configs/entailmentbank.yaml`
- Compare trajectory geometry of gold entailment chains (EntailmentBank) vs.
  model-generated chains (OpenBookQA) in `../studies/hidden_state_trajectories/`.